# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

Lane 4 (CTR / Engagement Opportunity Scoring). This notebook turns the validated output of
weeks 3–6 into a human-reviewed content action playbook: ranked actions with reason codes,
an archetype→action map, intended use, limits, the decay/refresh insight stated honestly,
human-review rules and a no-go list, monitoring/retrain triggers, and cost/value thinking.

The ranking engine is the **Week-4 hand rule** (`ctr_below_tier_benchmark`), kept over the
Week-5 model deliberately: five-fold client-grouped validation measured them equal
(P@50 0.952 ± 0.033 both), so the simpler, readable rule carries the queue.

## 0. Setup — rebuild the validated frame

In [ ]:
%pip -q install duckdb huggingface_hub pandas scikit-learn matplotlib

In [ ]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MAR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APR = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"

import pathlib

def find_repo_root():
    p = pathlib.Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'work' / 'notebooks').exists():
            return cand
    return p

ROOT = find_repo_root()
OUT_DIR = ROOT / 'work' / 'outputs'
FIG_DIR = ROOT / 'work' / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'outputs  -> {OUT_DIR}')
print(f'figures  -> {FIG_DIR}')

In [ ]:
import numpy as np
import pandas as pd

SEED = 42
LABEL = 'under_captured_apr'

q_mar = f"""
    WITH pagemo AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)  AS imp_mar,
               SUM(gsc_clicks)       AS clk_mar,
               AVG(gsc_avg_position) AS pos_mar
        FROM {MAR}
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100 AND AVG(gsc_avg_position) > 0
    ),
    climpo AS (
        SELECT client_hash_id, SUM(gsc_impressions) AS cli_imp
        FROM {MAR}
        GROUP BY 1
    )
    SELECT p.*, c.cli_imp
    FROM pagemo p JOIN climpo c USING (client_hash_id)
"""
mar = con.sql(q_mar).df()

q_apr = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_apr,
           SUM(gsc_clicks)      AS clk_apr
    FROM {APR}
    GROUP BY 1, 2
"""
apr = con.sql(q_apr).df()

def tier_of(pos):
    if pos <= 3:
        return 'p1_top'
    if pos <= 10:
        return 'p1'
    if pos <= 20:
        return 'p2'
    return 'deep'

queue = mar.copy()
queue['tier'] = queue['pos_mar'].apply(tier_of)
queue['ctr_mar'] = queue['clk_mar'] / queue['imp_mar']
bench = queue[queue['imp_mar'] >= 1000].groupby('tier')['ctr_mar'].median().rename('tier_expected_ctr')
queue['tier_expected_ctr'] = queue['tier'].map(bench)
bench_support = queue[queue['imp_mar'] >= 1000].groupby('tier').size().rename('bench_support')
queue['bench_support'] = queue['tier'].map(bench_support)
queue = queue[queue['tier_expected_ctr'] > 0].copy()

queue['capture_ratio'] = queue['ctr_mar'] / queue['tier_expected_ctr']
queue['rule_score'] = (100 * (1 - queue['capture_ratio']).clip(lower=0)
                       * np.log10(queue['imp_mar']))
queue = queue.sort_values(['client_hash_id', 'content_hash_id']).reset_index(drop=True)

# April is joined ONLY for evaluation receipts, never for ranking inputs
ev = queue.merge(apr, on=['client_hash_id', 'content_hash_id'], how='inner').copy()
ev['apr_ctr'] = np.where(ev['imp_apr'] > 0, ev['clk_apr'] / ev['imp_apr'], np.nan)
bench_apr = ev[ev['imp_apr'] >= 1000].groupby('tier')['apr_ctr'].median().rename('tier_expected_apr')
ev['tier_expected_apr'] = ev['tier'].map(bench_apr)
labeled = ev[(ev['imp_apr'] >= 100) & (ev['tier_expected_ctr'] > 0) &
             (ev['tier_expected_apr'] > 0)].copy()
labeled[LABEL] = ((labeled['apr_ctr'] / labeled['tier_expected_apr']) < 0.5).astype(int)

def p_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores, dtype=float))
    return float(np.asarray(labels)[order[:min(k, len(labels))]].mean())

base_rate_all = float(labeled[LABEL].mean())
rule_p50_eval = p_at_k(labeled['rule_score'], labeled[LABEL].to_numpy(), 50)

print(f'queue pool : {len(queue):,} visible page-months (Mar 2026)')
print(f'eval subset: {len(labeled):,} with April outcomes | label base rate {base_rate_all:.3f}')
print(f'replayed rule P@50 on eval subset: {rule_p50_eval:.3f}')

## 1. Ranked actions + reason codes

**Archetypes.** Every queued page falls into exactly one of four named archetypes, defined by
position tier × capture band. Each archetype carries a default action and an explicit
"what would change the action" clause — the mapping is a starting hypothesis for the analyst,
not a verdict.

| Archetype | Definition | Default action | What would change it |
|---|---|---|---|
| Page-One Underperformer | tier `p1_top`/`p1`, capture < 80% | rewrite title & meta | zero-click giant → instrumentation check before any edit |
| Striking-Distance Candidate | tier `p2`, capture < 80% | improve intent match | if position sits near a tier boundary, re-check after position settles |
| Deep Low-Yield | tier `deep`, capture < 80% | monitor only | meaningful position gain observed → re-archetype next cycle |
| Healthy — Leave Alone | capture ≥ 80% | no action this cycle | demand shift (impressions collapsing) → revisit |

**Reason codes.** One code per page, assigned mechanically from tier × capture × volume:

| Reason code | Condition | Action | Batch |
|---|---|---|---|
| `severe_undercapture_page1` | p1 tiers, ratio < 0.25 | `rewrite_title_meta` | P1 review |
| `moderate_undercapture_page1` | p1 tiers, 0.25 ≤ ratio < 0.8 | `rewrite_title_meta` | P2 review |
| `striking_distance_gap` | p2, ratio < 0.8 | `improve_intent_match` | P2 review |
| `deep_low_yield` | deep tier, ratio < 0.8 | `monitor_only` | monitor |
| `low_volume_manual_check` | impressions < 200 (overrides above) | tier action, human first | manual check |

The volume guard exists because Week-5's error analysis showed sub-200-impression pages are
where both the rule's and the model's confidence is least reliable (noisy March CTR).

In [ ]:
def archetype_of(r):
    if r['capture_ratio'] >= 0.8:
        return 'Healthy - leave alone'
    if r['tier'] in ('p1_top', 'p1'):
        return 'Page-one underperformer'
    if r['tier'] == 'p2':
        return 'Striking-distance candidate'
    return 'Deep low-yield'

def reason_of(r):
    if r['imp_mar'] < 200:
        return 'low_volume_manual_check'
    if r['tier'] in ('p1_top', 'p1'):
        return 'severe_undercapture_page1' if r['capture_ratio'] < 0.25 else 'moderate_undercapture_page1'
    if r['tier'] == 'p2':
        return 'striking_distance_gap'
    return 'deep_low_yield'

action_map = {
    'severe_undercapture_page1': ('rewrite_title_meta', 'P1 review'),
    'moderate_undercapture_page1': ('rewrite_title_meta', 'P2 review'),
    'striking_distance_gap': ('improve_intent_match', 'P2 review'),
    'deep_low_yield': ('monitor_only', 'monitor'),
    'low_volume_manual_check': None,  # resolved below from tier
}

def action_of(r):
    if r['reason_code'] == 'low_volume_manual_check':
        act = {'p1_top': 'rewrite_title_meta', 'p1': 'rewrite_title_meta',
               'p2': 'improve_intent_match', 'deep': 'monitor_only'}[r['tier']]
        return act, 'human check first'
    return action_map[r['reason_code']]

q = queue.copy()
q['archetype'] = q.apply(archetype_of, axis=1)
q['reason_code'] = q.apply(reason_of, axis=1)
q[['action_label', 'priority_batch']] = q.apply(
    lambda r: pd.Series(action_of(r)), axis=1)
q = q.sort_values('rule_score', ascending=False).reset_index(drop=True)
q.insert(0, 'rank', np.arange(1, len(q) + 1))

print('archetype mix:')
print(q['archetype'].value_counts().to_string())
print()
print('reason-code mix:')
print(q['reason_code'].value_counts().to_string())
print()
print('batch sizes:')
print(q['priority_batch'].value_counts().to_string())

In [ ]:
batch_mix = (q.groupby(['priority_batch', 'action_label'])
             .agg(n=('rank', 'size'), median_capture=('capture_ratio', 'median'))
             .round(3))
display(batch_mix)

top12 = q.head(12)[['rank', 'tier', 'pos_mar', 'imp_mar', 'capture_ratio',
                    'rule_score', 'reason_code', 'action_label', 'priority_batch']]
with pd.option_context('display.width', 220):
    display(top12.round(3))

## 2. Intended use and limits

**Intended use.** A weekly *review-session helper* for one content analyst: open the P1 review
batch, work down the ranked list within a fixed time budget (the Precision@K framing exists
because capacity is finite), apply the default action or overrule it, and log the decision.
Decision-support ordering only — the analyst owns every action taken.

**Where it is valid:** pages of clients with GSC history in this portfolio; visible pages
(≥100 monthly impressions); decisions made shortly after a month closes (features are
month-end aggregates); ranking quality as measured — rule P@50 ≈ 0.92–0.95 against a ~0.43–0.48
base rate on the March→April panel.

**Limits, each carried from earlier weeks with its number:**

1. **Survivorship (12.8%).** Pages that stop earning ≥100 April impressions never enter the
   labeled frame — the queue learns on survivors and says nothing about vanished pages.
2. **Saturated precision regime.** Disjoint top-50 lists scored alike (w05), so ordering
   *within* "worth reviewing" is directional, not ground truth.
3. **One step-ahead window.** Trained/validated on a single month pair; seasonality and
   algorithm-change effects are unmeasured here.
4. **Benchmark sparsity.** Tier benchmarks rest on same-month medians of ≥1,000-impression
   pages; rare tiers carry noisy bars (support counts travel with every row).
5. **GSC-only signal.** GA4 columns excluded by contract (zero-fill hazard); engagement-side
   judgments are out of scope for this queue.

In [ ]:
limits_receipt = {
    'queue_rows': int(len(q)),
    'eval_rows_with_outcome': int(len(labeled)),
    'survivorship_dropped_pct': round(100 * (1 - len(labeled) / len(queue)), 1),
    'label_base_rate_eval': round(base_rate_all, 3),
    'rule_p50_replay': round(rule_p50_eval, 3),
    'min_impressions_for_benchmark_pool': 1000,
    'volume_guard_threshold': 200,
}
display(pd.Series(limits_receipt).to_frame('value'))
limits_receipt

## 3. Human review + the no-go list

**Before acting on any pick, the analyst checks four things** (generated per row below):
1. **Instrumentation.** Is the near-zero CTR real, or a measurement artifact? (Top-ranked
   zero-click giants passed this check in w04's skeptic review.)
2. **Intent & SERP context.** Does the query surface features (snippets, PAA, shopping) that
   suppress CTR regardless of metadata quality?
3. **Cannibalization.** Does a sibling page split the same demand? (Merging may beat editing.)
4. **Sensitivity.** YMYL/legal/brand-critical content routes to a senior reviewer, full stop.

**The no-go list — what must never be automated, regardless of score:**
- No auto-publishing or auto-applying edits; every rewrite ships through normal editorial review.
- No page removals, redirects, or deindexing decided by model/rule scores alone.
- No cross-client bulk application: each client's queue is reviewed inside that client's context.
- No causal claims toward search algorithms ("Google rewards…") — the design is observational.
- No queue order treated as a quality ranking of pages; it ranks *review priority*, directionally.

In [ ]:
review = q[q['priority_batch'] == 'P1 review'].head(10).copy()

def check_for(r):
    checks = []
    if r['capture_ratio'] < 0.02:
        checks.append('verify zero-click is not an instrumentation artifact')
    near = min(abs(r['pos_mar'] - 3), abs(r['pos_mar'] - 10), abs(r['pos_mar'] - 20)) <= 0.5
    if near:
        checks.append('position near tier boundary - confirm archetype after position settles')
    if r['bench_support'] < 30:
        checks.append(f"benchmark rests on {int(r['bench_support'])} high-volume pages")
    if not checks:
        checks.append('standard intent/SERP + cannibalization check')
    return '; '.join(checks)

review['required_human_checks'] = review.apply(check_for, axis=1)
review['content_short'] = review['content_hash_id'].str[:12]
cols = ['rank', 'content_short', 'reason_code', 'action_label',
        'capture_ratio', 'rule_score', 'required_human_checks']
with pd.option_context('display.max_colwidth', 70, 'display.width', 230):
    display(review[cols].round(3))
print(f'P1 review batch size: {int((q["priority_batch"] == "P1 review").sum()):,} pages')

## 4. Monitoring / retrain triggers

Non-production by intent: a short list of measurable conditions that would tell us the
recommendations went stale, each with a named response. Thresholds are anchored to this run's
measured values so drift is defined relative to something concrete.

In [ ]:
median_imp_per_client = float(mar.groupby('client_hash_id')['imp_mar'].sum().median())

triggers = pd.DataFrame([
    {'trigger': 'new month partition lands',
     'measured anchor': 'monthly cadence',
     'threshold': 'each new closed month',
     'response': 'rebuild queue for the new feature month; recompute evaluation receipts'},
    {'trigger': 'label base-rate drift',
     'measured anchor': f'{base_rate_all:.3f} (Mar->Apr panel)',
     'threshold': 'outside +/-10% relative',
     'response': 'investigate population shift before trusting precision numbers'},
    {'trigger': 'rule ranking decays',
     'measured anchor': f'P@50 {rule_p50_eval:.3f} on replay',
     'threshold': '< 0.85 on the newest month-pair',
     'response': 're-validate; consider retraining LR and re-running the w06 audit'},
    {'trigger': 'portfolio composition shift',
     'measured anchor': f'median client monthly impressions {median_imp_per_client:,.0f}',
     'threshold': '> 30% change vs anchor',
     'response': 're-examine benchmark support and volume guard before shipping batches'},
    {'trigger': 'schema or availability change',
     'measured anchor': 'GA4-available share 4.2% (w03), column set fixed',
     'threshold': 'any column added/removed or availability share doubling',
     'response': 'freeze queue; re-run data-contract verification queries'},
])
with pd.option_context('display.max_colwidth', 60, 'display.width', 230):
    display(triggers)
print('Retraining itself stays manual and audited: any retrained model must repeat the')
print('week-6 protocol (grouped splits, sibling test, claim rewrite) before replacing the rule.')

## 5. Exports for the paper

Three artifacts land where the capstone paper builds on them:
`work/outputs/playbook_action_queue.csv` (regenerated every run, gitignored by design),
committed figure PNGs in `work/figures/`, and committed receipts in
`work/outputs/w07_playbook_metrics.json`.

In [ ]:
export_cols = ['rank', 'client_hash_id', 'content_hash_id', 'archetype', 'tier',
               'pos_mar', 'imp_mar', 'clk_mar', 'ctr_mar', 'tier_expected_ctr',
               'bench_support', 'capture_ratio', 'rule_score',
               'reason_code', 'action_label', 'priority_batch']
queue_csv = OUT_DIR / 'playbook_action_queue.csv'
q[export_cols].round(5).to_csv(queue_csv, index=False)
print(f'wrote {queue_csv} ({len(q):,} rows)')

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Fig 1: queue composition by archetype
fig1, ax1 = plt.subplots(figsize=(7, 4))
counts = q['archetype'].value_counts()
ax1.barh(counts.index[::-1], counts.values[::-1], color='#4472c4')
ax1.set_xlabel('pages')
ax1.set_title('Queue composition by archetype (Mar 2026, n=%s)' % f'{len(q):,}')
fig1.tight_layout()
f1 = FIG_DIR / 'fig1_archetype_mix.png'
fig1.savefig(f1, dpi=150)
plt.close(fig1)

# Fig 2: capture-ratio distribution by archetype (medians with quartiles)
fig2, ax2 = plt.subplots(figsize=(7, 4))
order = ['Page-one underperformer', 'Striking-distance candidate', 'Deep low-yield',
         'Healthy - leave alone']
data = [q.loc[q['archetype'] == a, 'capture_ratio'] for a in order]
ax2.boxplot(data, tick_labels=[a.replace(' - ', '\n') for a in order], showfliers=False)
ax2.axhline(0.5, ls='--', c='red', lw=1)
ax2.axhline(1.0, ls=':', c='gray', lw=1)
ax2.set_ylabel('capture ratio (CTR / tier benchmark)')
ax2.set_title('Capture ratio by archetype (red = under-capture bar, 0.5)')
fig2.tight_layout()
f2 = FIG_DIR / 'fig2_capture_by_archetype.png'
fig2.savefig(f2, dpi=150)
plt.close(fig2)

# Fig 3: precision@K curve, rule vs base rate (eval subset)
fig3, ax3 = plt.subplots(figsize=(7, 4))
ks = [10, 20, 50, 100, 200, 500]
scores = labeled['rule_score'].to_numpy()
y = labeled[LABEL].to_numpy()
prec = [p_at_k(scores, y, k) for k in ks]
ax3.plot(ks, prec, marker='o', label='rule (ctr_below_tier_benchmark)')
ax3.axhline(base_rate_all, ls='--', c='gray', lw=1,
            label=f'base rate ({base_rate_all:.2f})')
ax3.set_xscale('log')
ax3.set_xticks(ks)
ax3.set_xticklabels(ks)
ax3.set_xlabel('K (pages reviewed)')
ax3.set_ylabel('precision@K')
ax3.set_ylim(0, 1.05)
ax3.set_title('Review-precision curve vs random picking')
ax3.legend()
fig3.tight_layout()
f3 = FIG_DIR / 'fig3_precision_at_k.png'
fig3.savefig(f3, dpi=150)
plt.close(fig3)

from IPython.display import Image, display as ipy_display
for f in (f1, f2, f3):
    print(f'saved {f}')
    ipy_display(Image(filename=str(f)))

In [ ]:
import json, datetime, sklearn
metrics = {
    'notebook': 'w07_action_playbook.ipynb',
    'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'sklearn_version': sklearn.__version__,
    'seed': SEED,
    'ranking_engine': 'Week-4 hand rule (kept over LR: measured equal, simpler)',
    'queue_rows': int(len(q)),
    'batches': q['priority_batch'].value_counts().to_dict(),
    'reason_codes': q['reason_code'].value_counts().to_dict(),
    'archetypes': q['archetype'].value_counts().to_dict(),
    'eval_base_rate': round(base_rate_all, 3),
    'rule_p50_replay': round(rule_p50_eval, 3),
    'exports': {
        'queue_csv': 'work/outputs/playbook_action_queue.csv',
        'figures': ['work/figures/fig1_archetype_mix.png',
                    'work/figures/fig2_capture_by_archetype.png',
                    'work/figures/fig3_precision_at_k.png'],
    },
    'monitoring_anchors': {'base_rate': round(base_rate_all, 3),
                           'rule_p50_floor': 0.85,
                           'median_client_monthly_impressions': round(median_imp_per_client)},
}
metrics_path = OUT_DIR / 'w07_playbook_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'wrote {metrics_path}')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.